## Decorator

---

**Setup.** Let $X$ and $Y$ be sets. (A *set* is just a collection of allowed values: $X$ the inputs, $Y$ the outputs.) Write

$$Y^X \;:=\; \{\, f \mid f : X \to Y \,\}$$

for the **function space**. This is *itself* a set, and its elements are **whole functions** $f : X \to Y$, so a single function $f$ is one **point** of $Y^X$. (The power notation is literal: with finite sets there are $|Y|^{|X|}$ such functions, hence "$Y$ to the $X$." Calling $Y^X$ the **exponential object in $\mathbf{Set}$** simply names this construction inside $\mathbf{Set}$, the universe whose objects are sets and whose arrows are ordinary functions.) The thing being wrapped is one element of this space: $f \in Y^X$.

**Definition.** A **decorator** is an **endo-operator** on the function space. *Endo-* means *same space in, same space out*: a map carrying functions to functions of the **same** type,

$$\boxed{\,D : Y^X \to Y^X\,}\qquad\text{i.e.}\qquad D \in \operatorname{End}\!\big(Y^X\big),$$

where $\operatorname{End}(Y^X)$ denotes the set of all such same-in-same-out maps.

How do we actually *describe* one particular decorator $D$? We give a single rule saying what the wrapped function does when you finally call it. That rule is the **augmentation kernel** $h$. (*Augmentation* is the extra behavior we add; *kernel* is the one little rule that generates it.) The kernel takes **two inputs**, the original function $f$ and an argument $x$, and produces the final output:

$$h : Y^X \times X \longrightarrow Y, \qquad D(f) \;:=\; \big(\, x \mapsto h(f, x) \,\big).$$

Read the right-hand side as: *"$D(f)$ is the new function that, given an $x$, returns $h(f, x)$."* So $h$ is the **body** of the wrapped function, and it gets to see both $f$ and $x$.

**Why $D$ and $h$ are really the same thing (currying).** There look to be two objects here: the decorator $D$ (takes $f$, gives back a function) and the kernel $h$ (takes $f$ **and** $x$, gives back an output). *Currying* is the observation that they hold the **same information**, because you can turn either one into the other:

- **from $h$ to $D$:** fix $f$ but leave $x$ open, and the leftover "$x \mapsto h(f,x)$" *is* $D(f)$;
- **from $D$ to $h$:** feed $D$ a function $f$, then feed *that result* an $x$, and the output is $h(f,x)$.

Nothing is lost either way, so as sets the two collections match up perfectly. (The "$\cong$" means *the same, up to a faithful renaming*.)

$$\underbrace{\operatorname{End}\!\big(Y^X\big)}_{\text{all } D:\,Y^X \to Y^X} \;=\; \big(Y^X\big)^{Y^X} \;\;\cong\;\; \underbrace{Y^{\,(Y^X \times X)}}_{\text{all } h:\,Y^X \times X \to Y}.$$

The upshot: **to specify a decorator you only ever write its kernel $h$, and the decorator $D$ is then automatic.** The two sides are joined by one evaluation rule,

$$D(f)(x) \;=\; h(f, x).$$

And notice that $h$ *receives* $f$ as an argument; it is handed $f$, not forced to run it first. So the kernel may call $f$ **zero** times (a cache hit returns a stored value), **once** (the usual case), or **many** times (a retry). That freedom is exactly what separates a decorator from plain post-composition $a \circ f$, which would always run $f$ exactly once.

**Stacking $=$ the monoid $\operatorname{End}(Y^X)$.** A **monoid** is a set with an associative way to combine two elements plus a do-nothing element. Endo-operators supply exactly this: they **compose**, composition is associative, and $\operatorname{id}_{Y^X}$ is a unit, so

$$\big(\operatorname{End}(Y^X),\ \circ,\ \operatorname{id}_{Y^X}\big)$$

is a monoid. A stack of decorators is a product in it:

$$D_3 \circ D_2 \circ D_1 \;\in\; \operatorname{End}(Y^X), \qquad (D_3 \circ D_2 \circ D_1)(f) \;=\; D_3\big(D_2(D_1(f))\big).$$

The unit $\operatorname{id}_{Y^X}$ (the **identity**, satisfying $\operatorname{id}(f) = f$) is the **trivial decorator**: it adds nothing.

**The three conditions, restated structurally.**

1. **Type preservation.** Automatic: $D$ is *endo* on $Y^X$, so $D(f) \in Y^X$ by construction. Same domain $X$, same codomain $Y$; nothing to verify.
2. **Non-modification.** $D$ does not mutate its argument: $f$ stays a fixed element of $Y^X$, and $D(f)$ is a *new* element built around it.
3. **Composability.** This is exactly the monoid structure on $\operatorname{End}(Y^X)$: stacking is closed, associative, and unital.

**Many methods $=$ a product of function spaces.** A real interface exposes *several* methods $g_i : X_i \to Y_i$ at once. Bundling them, the whole interface is one point of the **product**

$$I \;=\; \prod_i Y_i^{\,X_i}$$

(a tuple, one coordinate per method), and a decorator of that interface is an endo-operator $D : I \to I$, the single-function theory above applied coordinate-wise.

### The common shape: before / after hooks

When the kernel applies $f$ **exactly once**, it factors through a *pre* map $b : X \to X$ and a *post* map $a : Y \to Y$:

$$h(f, x) = a\big(f(b(x))\big), \qquad\text{i.e.}\qquad D(f) = a \circ f \circ b.$$

| Kernel does | $D(f)$ | Example |
|---|---|---|
| post-process only | $a \circ f$ &nbsp;($b = \operatorname{id}_X$) | "double the result" |
| pre-process only | $f \circ b$ &nbsp;($a = \operatorname{id}_Y$) | validate / log the input |
| skip $f$ sometimes | does not factor; needs full $h(f,x)$ | caching, auth guard, retry |

The narrow form $D(f) = a \circ f$ is only the first row; keeping $f$ inside $h$ is what buys the third.


### Exercise 1 — Logging Decorator

---

**Scenario:** A `TextEditor` has `write(text)`. You want to log every call **without touching `TextEditor`**. The decorator wraps it, adding logging as the augmentation $h$.

**Your task:** Write a `LoggingDecorator` wrapping any text editor. It logs **before** every `write()` call, then delegates to $f$.

```python
editor = LoggingDecorator(TextEditor())   # D(f)
editor.write("Hello")
# [LOG] write() called with: Hello        <- h (the added behavior)
# Hello                                    <- f(x) (the original)
```

**Hints**

- The decorator holds `self._editor = editor` — this stores $f$. Its `write()` is the decorated call: print the log ($h$), **then** call `self._editor.write(text)` ($f$). Since the log runs *before* $f$, this is the **pre-hook** shape $D(f)(x) = f(b(x))$ where the "$b$" step is the logging side-effect.
- The decorator must expose the **same** interface as `TextEditor` — same method name `write`. This is the **type preservation** condition: $D(f) : X \rightarrow Y$, so the client can't tell it's talking to a decorator.


In [5]:
#--------------------------------
# Original (f) — you cannot change this

class TextEditor:
    def write(self, text):
        print(text)

#--------------------------------
# Decorator (D) — your task: log before delegating to f
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class LoggingDecorator:
    def __init__(self, editor):
        self._editor = editor           # stores f

    def write(self, text):              # same interface as TextEditor (type preservation)
        # 1) log the call
        # 2) self._editor.write(text)                (f(x))
        logger.info(f"[LOG] write() called with: {text}")
        return self._editor.write(text)
        

#--------------------------------
editor = LoggingDecorator(TextEditor())   # D(f)
editor.write("Hello")
# expected:
# write() called with: Hello
# Hello


INFO:__main__:[LOG] write() called with: Hello


Hello


### Exercise 2 — Stacked Decorators (Coffee Shop)

---

**Scenario:** A `Coffee` has `cost()` and `description()`. Add-ons (`Milk`, `Sugar`, `Vanilla`) each increment the cost and extend the description — and can be **stacked in any order**.

**Your task:** Build add-on decorators that compose freely: $D_{\text{Vanilla}}\big(D_{\text{Milk}}(D_{\text{Sugar}}(f))\big)$.

```python
drink = Vanilla(Milk(Sugar(Coffee())))
print(drink.cost())         # sum of all layers
print(drink.description())  # Coffee, Sugar, Milk, Vanilla
```

**Hints**

- Each add-on stores the inner drink: `self._drink = drink`. Then `cost()` returns `self._drink.cost() + self.added_cost` — this is $h(f(x))$ where $h$ adds the increment ($f$ = the inner drink, untouched).
- Swap the wrapping order: the **total cost stays the same**, only the **description order changes**. This confirms the **composability** condition $D_3(D_2(D_1(f)))$.


In [ ]:
#--------------------------------
# Original (f) — the base drink, you cannot change this

class Coffee:
    def cost(self):
        return 2.0

    def description(self):
        return "Coffee"

#--------------------------------
# Add-on decorators (D) — each wraps a drink and adds to cost + description
# Every add-on exposes the SAME interface: cost() and description()  (type preservation)

class Sugar:
    added_cost = 0.25
    def __init__(self, drink):
        self._drink = drink                         # stores f (inner drink)
    def cost(self):
        return self._drink.cost() + self.added_cost # self._drink.cost() + self.added_cost
                                         
    def description(self):
        return self._drink.description() + ", Sugar" # self._drink.description() + ", Sugar"

class Milk:
    added_cost = 0.50
    def __init__(self, drink):
        self._drink = drink
    def cost(self):
        return self._drink.cost() + self.added_cost

    def description(self):
        return self._drink.description() + ", Milk"

class Vanilla:
    added_cost = 0.75
    def __init__(self, drink):
        self._drink = drink
    def cost(self):
        return self._drink.cost() + self.added_cost

    def description(self):
        return self._drink.description() + ", Vanilla"

#--------------------------------
drink = Vanilla(Milk(Sugar(Coffee())))              # D_Vanilla(D_Milk(D_Sugar(f)))
print(drink.cost())                                 # expected: 3.5
print(drink.description())                          # expected: Coffee, Sugar, Milk, Vanilla

# composability check — different order, SAME cost, different description order:
other = Sugar(Vanilla(Milk(Coffee())))
print(other.cost())                                 # expected: 3.5  (unchanged)
print(other.description())                          # expected: Coffee, Milk, Vanilla, Sugar


3.5
Coffee, Sugar, Milk, Vanilla
3.5
Coffee, Milk, Vanilla, Sugar
